# Experimentation with GEC into the full pipeline


In [6]:
import json
import logging
import os
import sys

# NOTE: import from root of the project
sys.path.append(os.path.abspath("../../../../"))

from src.services.gec.modules.dictionary.arramooz_client import ArramoozClient
from src.services.gec.modules.dictionary.engine import DictionaryEngine
from src.services.gec.modules.ontology.engine import OntologyEngine
from src.services.gec.schemas import (
    DictionaryCandidateEdit,
    GECInput,
    ModuleStatus,
    OntologyCandidateEdit,
)
from src.services.preprocessing.orchestrator import preprocess
from src.services.preprocessing.schemas import PreprocessingInput

logging.basicConfig(level=logging.DEBUG)
logging.getLogger("src.services.gec.modules.dictionary.spell_checker").setLevel(
    logging.DEBUG
)

In [7]:
arramooz_client = ArramoozClient()
dictionary_engine = DictionaryEngine(arramooz_client)
ontology_engine = OntologyEngine(arramooz_client)

2026-06-25 04:08:25.726 | INFO     | src.services.gec.modules.dictionary.arramooz_client:__init__:36 - ArramoozClient initialized | dict_db=/Users/incorta/College/GP/baligh/src/services/gec/data/dictionary/arabicdictionary.sqlite
2026-06-25 04:08:25.842 | INFO     | src.services.gec.modules.dictionary.arramooz_client:get_all_normalized_words:123 - Fetched 31190 normalized words from dictionary
2026-06-25 04:08:25.862 | INFO     | src.services.gec.modules.dictionary.spell_checker:__init__:28 - SpellChecker initialized | vocabulary_size=31190
2026-06-25 04:08:25.863 | INFO     | src.services.gec.modules.dictionary.engine:__init__:37 - DictionaryEngine initialized successfully
2026-06-25 04:08:25.867 | DEBUG    | src.services.gec.modules.ontology.loader:__new__:32 - Returning existing OntologyLoader singleton instance
2026-06-25 04:08:25.868 | DEBUG    | src.services.gec.modules.ontology.loader:load_graph:63 - Ontology graph already loaded; skipping re-parse
2026-06-25 04:08:25.933 | INFO

In [8]:
def test_dictionary(
    text: str, sum_output: bool = True, show_preprocessing: bool = False
):
    """Tests dictionary-based GEC with preprocessing."""
    pre_input = PreprocessingInput(text=text)
    pre_output = preprocess(pre_input)

    gec_input = GECInput(
        text=pre_output.text,
        tokens=pre_output.tokens,
        morph_features=pre_output.morph_features,
        errors_span=[],
    )
    gec_output = dictionary_engine.process(gec_input)

    if show_preprocessing:
        print("Preprocessing output:")
        print(json.dumps(pre_output.model_dump(), ensure_ascii=False, indent=2))

    if sum_output:
        if gec_output.status == ModuleStatus.CORRECT:
            print("No errors found")
        else:
            print(f"Found {len(gec_output.candidate_edits)} candidate edits:")
            for i in range(len(gec_output.candidate_edits)):
                edit = gec_output.candidate_edits[i]
                alternatives = sorted(edit.alternatives or [], reverse=True)
                error_tokens = pre_output.tokens[
                    edit.token_refs[0] : edit.token_refs[-1] + 1
                ]
                print("*" * 20 + f" Edit {i} " + "*" * 20)
                print(f"  Error in: {error_tokens}")
                print(f"  Corrected text: {edit.correction}")
                print(f"  Alternative texts: {alternatives}")
    else:
        print(json.dumps(gec_output.model_dump(), ensure_ascii=False, indent=2))

In [9]:
def test_ontology(text: str, sum_output: bool = True, show_preprocessing: bool = False):
    """Tests ontology-based GEC with preprocessing."""
    pre_input = PreprocessingInput(text=text)
    pre_output = preprocess(pre_input)

    gec_input = GECInput(
        text=pre_output.text,
        tokens=pre_output.tokens,
        morph_features=pre_output.morph_features,
        errors_span=[],
    )
    gec_output = ontology_engine.process(gec_input)

    if show_preprocessing:
        print("Preprocessing output:")
        print(json.dumps(pre_output.model_dump(), ensure_ascii=False, indent=2))

    if sum_output:
        if gec_output.status == ModuleStatus.CORRECT:
            print("No errors found")
        else:
            print(f"Found {len(gec_output.candidate_edits)} candidate edits:")
            for i in range(len(gec_output.candidate_edits)):
                edit = gec_output.candidate_edits[i]
                error_tokens = pre_output.tokens[
                    edit.token_refs[0] : edit.token_refs[-1] + 1
                ]
                print("*" * 20 + f" Edit {i} " + "*" * 20)
                print(f"  Error in: {error_tokens}")
                print(f"  Corrected text: {edit.correction}")
                print(f"  Explanation: {edit.explanation}")
    else:
        print(json.dumps(gec_output.model_dump(), ensure_ascii=False, indent=2))

---
### Open testing

In [12]:
text = " قرأ الطالبين الكتابان "
# test_dictionary(text, sum_output=True)
test_ontology(text, sum_output=True)

2026-06-25 04:11:02.182 | INFO     | src.services.gec.modules.ontology.engine:process:47 - tokens=3 errors_span=0
2026-06-25 04:11:02.182 | INFO     | src.services.gec.modules.ontology.candidate_generator:generate_candidates:67 - tokens=3 spans=0
2026-06-25 04:11:02.653 | DEBUG    | src.services.gec.modules.ontology.candidate_generator:generate_candidates:80 - Discovered relations: 8
2026-06-25 04:11:02.653 | DEBUG    | src.services.gec.modules.ontology.loader:query:80 - Executing SPARQL query
2026-06-25 04:11:02.696 | DEBUG    | src.services.gec.modules.ontology.loader:query:80 - Executing SPARQL query
2026-06-25 04:11:02.746 | DEBUG    | src.services.gec.modules.ontology.loader:query:80 - Executing SPARQL query
2026-06-25 04:11:02.798 | DEBUG    | src.services.gec.modules.ontology.loader:query:80 - Executing SPARQL query
2026-06-25 04:11:02.851 | DEBUG    | src.services.gec.modules.ontology.loader:query:80 - Executing SPARQL query
2026-06-25 04:11:02.909 | DEBUG    | src.services.gec

Found 1 candidate edits:
******************** Edit 0 ********************
  Error in: [Token(index=0, form='قرأ', span=(1, 4), norm_span=(1, 4), affix_structure='STEM', farasa_segmentation='قرأ', is_oov=False), Token(index=1, form='الطالبين', span=(5, 13), norm_span=(5, 13), affix_structure='DET+STEM', farasa_segmentation='ال+طالب+ين', is_oov=False), Token(index=2, form='الكتابان', span=(14, 22), norm_span=(14, 22), affix_structure='DET+STEM', farasa_segmentation='ال+كتاب+ان', is_oov=False)]
  Corrected text: قرأ الطالبان الكتابين
  Explanation: الفاعل يجب أن يكون مرفوعاً ولكن وجد مجروراً
